# Blueprint from JSON (Visual Builder)

Загрузка Blueprint из файла (Visual Model Builder), сборка `BlueprintForCausalLM`, forward и подсчёт параметров.

In [ ]:
from pathlib import Path
import torch
from homellm.models.blueprint import Blueprint
from homellm.models.blueprint_model import BlueprintLMConfig, BlueprintForCausalLM

BLUEPRINT_DIR = Path('/app/blueprints')
jsons = list(BLUEPRINT_DIR.glob('*.json'))
BP_PATH = jsons[0] if jsons else BLUEPRINT_DIR / 'model.json'
print('Blueprint path:', BP_PATH)

In [ ]:
bp = Blueprint.load(BP_PATH)
print('vocab_size:', bp.vocab_size, 'hidden_size:', bp.hidden_size, 'blocks:', len(bp.blocks))

In [ ]:
config = BlueprintLMConfig(
    vocab_size=bp.vocab_size,
    hidden_size=bp.hidden_size,
    max_position_embeddings=bp.max_position_embeddings,
    auto_project=bp.auto_project,
    blueprint=bp.dict(),
)
model = BlueprintForCausalLM(config)
total = sum(p.numel() for p in model.parameters())
print('Params:', total)

In [ ]:
model.eval()
x = torch.randint(0, bp.vocab_size, (1, 32))
with torch.no_grad():
    out = model(input_ids=x, labels=x)
print('Forward OK. Loss:', out.loss.item())